In [1]:
cd ..


/home/karaman/Documents/bet


In [2]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import src.utils.config as config
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import mlflow
import json
from mlflow.models.signature import infer_signature
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import sys
import platform
import subprocess
import hashlib



In [3]:

def compute_metrics(df, threshold):
    pred = df.withColumn(
        "pred_label",
        (F.col("p_churn") >= threshold).cast("int")
    )
    metrics = pred.groupBy("next_7d_churn_idx", "pred_label").count()
    tp = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 1").select("count").first()
    fp = metrics.filter("next_7d_churn_idx = 0 AND pred_label = 1").select("count").first()
    fn = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 0").select("count").first()

    tp = tp[0] if tp else 0
    fp = fp[0] if fp else 0
    fn = fn[0] if fn else 0

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return precision, recall, f1

def add_class_weight(df, weight_for_churn):
    return df.withColumn(
        "class_weight",
        F.when(F.col("next_7d_churn"), weight_for_churn).otherwise(1.0)
    )


In [4]:

spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

sample_fraction = 0.2
mlflow.set_experiment("first experiment")
experiment_tags = {
    "data_sample_fraction": sample_fraction, 
    "data_scope": "sampled",
    }

with open("experiment_tags.json", "w") as f:
    json.dump(experiment_tags, f, indent=2)

player_snapshot = player_snapshot.select(
'player_idx', 'country', 'age_bucket', 
#'device_type', #'acquisition_channel', #'registration_date', #'risk_segment',
  )

sample_players = player_snapshot.select("player_idx").sample(sample_fraction)
model_df = (
    player_behavior
        .join(player_snapshot, on="player_idx", how="left")
        .join(labels, on=["player_idx", "reference_date"], how="inner")
)

sample_dataset = sample_players.join(model_df, on="player_idx", how="inner") \
                               .withColumn("next_7d_churn_idx", F.col("next_7d_churn").cast("int"))


numeric_cols = [
    "balance_7d_ago", "balance_30d_ago", "net_amount_result_7d",
    "net_amount_result_30d", "num_sessions_7d", "num_sessions_30d",
    "avg_sessions_duration_30d", "avg_bet_amount_30d",
    "net_game_result_7d", "net_game_result_30d",
    "failed_withdrawals_30d", "deposit_count_30d", "withdrawal_count_30d",
    "withdrawal_ratio"
]

categorical_cols = ["country", "age_bucket"]
categorical_idx = [c + "_idx" for c in categorical_cols]
categorical_ohe = [c + "_ohe" for c in categorical_cols]

indexer = StringIndexer(inputCols=categorical_cols, outputCols=categorical_idx, handleInvalid="error")
ohe = OneHotEncoder(inputCols=categorical_idx, outputCols=categorical_ohe, dropLast=False)

numeric_assembler = VectorAssembler(inputCols=numeric_cols, outputCol="numeric_features")
scaler = StandardScaler(inputCol="numeric_features", outputCol="numeric_features_scaled", withMean=True, withStd=True)

final_assembler = VectorAssembler(inputCols=["numeric_features_scaled"] + categorical_ohe, outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="next_7d_churn_idx", weightCol="class_weight", maxIter=50)

pipeline = Pipeline(stages=[indexer, ohe, numeric_assembler, scaler, final_assembler, lr])

evaluator = BinaryClassificationEvaluator(labelCol="next_7d_churn_idx", metricName="areaUnderPR")

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

cv = CrossValidator(estimator=pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3, parallelism=2)

feature_metadata = {
    "numeric_features": numeric_cols,
    "categorical_features": categorical_cols,
    "label": "next_7d_churn",
    "label_definition": "completion of 7-day inactivity window within next 7 days"
}

with open("feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=2)


    




Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/09 11:38:59 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.1.179 instead (on interface ens33)
26/02/09 11:38:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/09 11:39:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026/02/09 11:39:06 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/09 11:39:06 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/09 11:39:06 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/09 11:39:06 INFO alembic.r

In [5]:

# ---------------------------
# Train / Val / Test Split
# ---------------------------

dates = [row.reference_date for row in sample_dataset.select("reference_date").distinct().orderBy("reference_date").collect()]
n = len(dates)
train_cut = dates[int(n * 0.70)]
val_cut = dates[int(n * 0.85)]

train_df = sample_dataset.filter(F.col("reference_date") < train_cut)
val_df = sample_dataset.filter((F.col("reference_date") >= train_cut) & (F.col("reference_date") < val_cut))
test_df = sample_dataset.filter(F.col("reference_date") >= val_cut)

# Class weighting
num_churn = train_df.filter("next_7d_churn = true").count()
num_nonchurn = train_df.filter("next_7d_churn = false").count()
weight_for_churn = num_nonchurn / num_churn

train_df = add_class_weight(train_df, weight_for_churn)
val_df = add_class_weight(val_df, weight_for_churn)
test_df = add_class_weight(test_df, weight_for_churn)



In [7]:
with mlflow.start_run(run_name='train') as run:
    mlflow.log_artifact("experiment_tags.json")

    # ---------------- DATA METADATA ----------------
    mlflow.log_param("train_start", str(train_df.agg(F.min("reference_date")).first()[0]))
    mlflow.log_param("train_end",   str(train_df.agg(F.max("reference_date")).first()[0]))
    mlflow.log_param("val_start",   str(val_df.agg(F.min("reference_date")).first()[0]))
    mlflow.log_param("val_end",     str(val_df.agg(F.max("reference_date")).first()[0]))
    mlflow.log_param("test_start",  str(test_df.agg(F.min("reference_date")).first()[0]))
    mlflow.log_param("test_end",    str(test_df.agg(F.max("reference_date")).first()[0]))

    mlflow.log_param("python_version", sys.version)
    mlflow.log_param("platform", platform.platform())


    try:
        git_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.STDOUT
        ).decode().strip()
    except Exception as e:
        git_commit = "unknown"
        git_error = str(e)

    mlflow.log_param("git_commit", git_commit)

    try:
        git_branch = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"]
        ).decode().strip()
    except Exception:
        git_branch = "unknown"

        mlflow.log_param("git_branch", git_branch)


    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("val_rows", val_df.count())
    mlflow.log_param("test_rows", test_df.count())

    mlflow.log_param("num_features", len(numeric_cols) + len(categorical_ohe))

    mlflow.log_param("num_churn_train", num_churn)
    mlflow.log_param("num_nonchurn_train", num_nonchurn)
    mlflow.log_param("class_weight", weight_for_churn)

    cv_model = cv.fit(train_df)
    best_model = cv_model.bestModel
    lr_best_model = best_model.stages[-1]

    mlflow.log_param("elasticParam", lr_best_model.getElasticNetParam())
    mlflow.log_param("regParam", lr_best_model.getRegParam())

    train_preds = best_model.transform(train_df).withColumn("p_churn", vector_to_array("probability")[1])
    train_upr = evaluator.evaluate(train_preds)
    mlflow.log_metric("upr", train_upr)

    coeffs = lr_best_model.coefficients.toArray().tolist()
    feature_names = numeric_cols + categorical_ohe
    ohe_model = best_model.stages[1]  # adjust index for your pipeline

    expanded_features = []

    # Add numeric features first
    expanded_features.extend(numeric_cols)

    # For each categorical column
    for input_col, output_col, category_sizes in zip(categorical_cols, categorical_ohe, ohe_model.categorySizes):
        for i in range(category_sizes):
            expanded_features.append(f"{input_col}_{i}")  # e.g., country_0, country_1, ...

    fi = pd.DataFrame({ "feature": expanded_features,  "coefficient": coeffs  })
    fi.to_csv("feature_importance.csv", index=False)
    mlflow.log_artifact("feature_importance.csv")


    sample_input = train_df.limit(100).toPandas()
    sample_output = train_preds.select("p_churn").limit(100).toPandas()
    signature = infer_signature(sample_input, sample_output)

    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path='spark_model',
        registered_model_name='SparkLogisticRegression',
        signature=signature
    )

    train_run_id = run.info.run_id

model_uri = f"runs:/{train_run_id}/spark_model"
loaded_model = mlflow.spark.load_model(model_uri)


/home/karaman/Documents/bet/venv/lib/python3.13/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'SparkLogisticRegression' already exists. Creating a new version of this model...
Created version '4' of model 'SparkLogisticRegression'.
2026/02/09 11:44:56 INFO mlflow.spark: URI 'runs:/4204e326fa504d9c89b53

In [8]:

with mlflow.start_run(run_name='val') as run:
    mlflow.set_tag("train_run_id", train_run_id)
    val_preds = loaded_model.transform(val_df).withColumn("p_churn", vector_to_array("probability")[1])
    val_upr = evaluator.evaluate(val_preds)
    mlflow.log_metric("upr", val_upr)


In [19]:

thresholds = [i / 100 for i in range(5, 96, 5)]
val_preds.persist()
val_preds.count()

with mlflow.start_run(run_name='thresholds'):
    mlflow.set_tag("train_run_id", train_run_id)
    results = []
    
    for t in thresholds:
        #with mlflow.start_run(nested=True, run_name=f"thr_{t}"):
        precision, recall, f1 = compute_metrics(val_preds, t)
        results.append({'threshold': t, 'precision':precision, 'recall':recall, 'f1':f1})
        # mlflow.log_param("threshold", t)
        # mlflow.log_metric("f1", f1)
        # mlflow.log_metric("recall", recall)
        # mlflow.log_metric("precision", precision)

    pd.DataFrame(results).to_csv("threshold_metrics.csv", index=False)
    mlflow.log_artifact("threshold_metrics.csv")    
    
    with open ('threshold_metrics.json', 'w') as f :
        json.dump(results, f, indent=2)
    mlflow.log_artifact("threshold_metrics.json")

metrics_df = spark.createDataFrame(  results)
best_threshold = metrics_df.orderBy(F.desc("f1")).first()["threshold"]

val_preds.unpersist()


DataFrame[player_idx: bigint, reference_date: date, balance_7d_ago: double, balance_30d_ago: double, net_amount_result_7d: double, net_amount_result_30d: double, num_sessions_7d: bigint, net_game_result_7d: double, num_sessions_30d: bigint, avg_sessions_duration_30d: double, avg_bet_amount_30d: double, net_game_result_30d: double, failed_withdrawals_30d: bigint, deposit_count_30d: bigint, withdrawal_count_30d: bigint, withdrawal_ratio: double, country: string, age_bucket: string, next_7d_churn: boolean, next_7d_churn_idx: int, class_weight: double, country_idx: double, age_bucket_idx: double, country_ohe: vector, age_bucket_ohe: vector, numeric_features: vector, numeric_features_scaled: vector, features: vector, rawPrediction: vector, probability: vector, prediction: double, p_churn: double]

In [20]:
best_threshold

0.75

In [21]:

with mlflow.start_run(run_name='test'):
    
    mlflow.set_tag("train_run_id", train_run_id)
    test_preds = loaded_model.transform(test_df).withColumn("p_churn", vector_to_array("probability")[1])
    test_upr = evaluator.evaluate(test_preds)

    precision, recall, f1 = compute_metrics(test_preds, best_threshold)
    mlflow.log_param('threshold', best_threshold)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('precision', precision)
    mlflow.log_metric("upr", test_upr)
    
    cm = (
    test_preds
    .withColumn("pred_label", (F.col("p_churn") >= best_threshold).cast("int"))
    .groupBy("next_7d_churn_idx", "pred_label")
    .count()
    )

    cm_pd = cm.toPandas()
    cm_pd.to_csv("confusion_matrix.csv", index=False)
    mlflow.log_artifact("confusion_matrix.csv")


    pdf = test_preds.select("p_churn", "next_7d_churn_idx").toPandas()

    precision_arr, recall_arr, _ = precision_recall_curve(
        pdf["next_7d_churn_idx"],
        pdf["p_churn"]
    )

    plt.figure()
    plt.plot(recall_arr, precision_arr)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.savefig("pr_curve.png")
    plt.close()

    mlflow.log_artifact("pr_curve.png")












In [30]:
test_preds.withColumn("pred_label", (F.col("p_churn") >= best_threshold).cast("int")).groupBy("pred_label",'next_7d_churn_idx').count().show()




+----------+-----------------+------+
|pred_label|next_7d_churn_idx| count|
+----------+-----------------+------+
|         1|                0|  6816|
|         1|                1| 19995|
|         0|                0|247074|
|         0|                1| 14747|
+----------+-----------------+------+



In [31]:
(test_preds
    .withColumn("pred_label", (F.col("p_churn") >= best_threshold).cast("int"))
    .groupBy("next_7d_churn_idx", "pred_label")
    .count()
    )

DataFrame[next_7d_churn_idx: int, pred_label: int, count: bigint]